# NACC data importation, analysis and preparation

## Libraries and Data Files
- Import necessary libraries, `pandas` and `numpy`.
- Load several CSV files containing data and metadata for analysis.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
# all_nacc = pd.read_csv('../../data/investigator_ftldlbd_nacc65.csv')
all_nacc = pd.read_csv('../../data/new_nacc_unique_type_3.csv')
nacc_variable = pd.read_csv('../../data/nacc_variable.csv')
nacc_allowable_code = pd.read_csv('../../data/nacc_allowable_code.csv')

## Data Cleaning

- Fill missing values in variable_id column using forward filling method.

In [3]:
nacc_allowable_code['variable_id_conv'] = nacc_allowable_code['variable_id'].fillna(method='ffill')
nacc_allowable_code.rename(columns={"descriptor": "individual_descriptor"}, inplace=True)

## Data Merging

- Merge `nacc_variable` with `nacc_allowable_code` on variable IDs to consolidate variable descriptions and their corresponding codes.
- Filter merged data for specific UDS forms ('A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C1', 'C1, C2', 'C1, C2, C2T', 'C1,C2', 'C2, C2T', 'C2', 'C2T', 'G1', 'D2) to get clinical features.
- Filter merged data for specific UDS forms (B4, B9 and D1) to get diagnosis features.

In [4]:
form = {
    'A1': '(Subject Demographics)',
    'A2': '(Co-participant Demographics)',
    'A3': '(Subject Family History)',
    'A4': '(Subject Medications)',
    'A5': '(Subject Health History)',
    'B1': '(Physical)',
    'B2': '(His and CVD)',
    'B3': "(Unified Parkinson's Disease Rating Scale (UPDRS))",
    'B4': '(Clinical Dementia Rating (CDR))',
    'B5': '(Neuropsychiatric Inventory Questionnaire (NPI-Q))',
    'B6': '(Geriatric Depression Scale (GDS))',
    'B7': '(Functional Assessment Scale (FAS))',
    'B8': '(Physical/ Neurological Exam Findings)',
    'B9': '(Clinician Judgment of Symptoms)',
    'C': '(Neuropsychological battery Summary Scores)',
    'D1': '(Clinical Diagnosis)',
    'D2': '(Clinician-assessed Medical Conditions)',
    'G1': '(Genetic testing)',
}

In [5]:
merged_df = pd.merge(nacc_variable, nacc_allowable_code, left_on='id', right_on='variable_id_conv', how='outer')
form_id_desc = []
for i in list(merged_df['form_id']):
    try:
        form_id_desc.append(form[i])
    except:
        form_id_desc.append(np.NaN)
merged_df['form_id_desc'] = form_id_desc
merged_df = merged_df.sort_values('form_id')

forms = ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C', 'G1', 'D2']
merged_df_patient = merged_df[(merged_df['form_id'].isin(forms))]
merged_df_diag = merged_df[merged_df['form_id'].isin(['B9', 'D1'])]

## Data Transformation

- Generate dictionaries for 
    - variable name and text diagnosis
    - descriptor codes and text descriptions for each code.

In [6]:
diag_dict = dict(zip(merged_df_diag['id'], zip(merged_df_diag['form_id_desc'], merged_df_diag['descriptor'])))
code_dict = {
    id_value: {row['code_1']: row['individual_descriptor'] for index, row in group.iterrows()}
    for id_value, group in merged_df_diag.groupby('id')
}

In [7]:
patient_dict = dict(zip(merged_df_patient['id'], zip(merged_df_patient['form_id_desc'], merged_df_patient['descriptor'])))
patient_code_dict = {
    id_value: {
        (row['code_1'] if pd.isna(row['code_2']) else "range"):
        (row['individual_descriptor'] if pd.isna(row['code_2']) else (int(row['code_1']), int(row['code_2'])))
        for index, row in group.iterrows()
    }
    for id_value, group in merged_df_patient.groupby('id')
}

In [8]:
patient_code_dict['ANIMALS']

{'range': (0, 77),
 '95': 'Physical problem',
 '96': 'Cognitive/behavior problem',
 '97': 'Other problem',
 '98': 'Verbal refusal',
 '-4': 'Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question'}

## Patient Record Dictionary Creation

- Filter the record columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding record texts.

In [9]:
patient_dict.update({f"DRUG{i}" : patient_dict["DRUG1-DRUG40"] for i in range(1,41)})
patient_dict = {k: v for k, v in patient_dict.items() if k in all_nacc.columns}
filtered_nacc = all_nacc[list(patient_dict.keys()) + ["NACCID"]].replace({-4: np.NaN, "-4": np.NaN})

In [10]:
from tqdm import tqdm

patient_texts = []
for i, row, in tqdm(filtered_nacc.iterrows()):
    st = ""
    for k, v in dict(row).items():
        if k not in row:
            continue
        if k == "NACCID":
            continue
        val = patient_dict[k][0] + patient_dict[k][1]
        try:
            if not pd.isna(v):
                if str(int(v)) in patient_code_dict[k]:
                    st += f"{val.replace(';', ',')}: {patient_code_dict[k][str(int(v))].replace(';', ',')}; "
                elif "range" in patient_code_dict[k] and int(v) in range(patient_code_dict[k]["range"][0], patient_code_dict[k]["range"][1]):
                    st += f"{val.replace(';', ',')}: {v}; "
        except:
            st += f"{val.replace(';', ',')}: {v}; "
        
    patient_texts.append(st)

47165it [01:11, 659.42it/s]


In [11]:
len(patient_texts)

47165

## Diagnosis Dictionary Creation

- Filter the diagnostic columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding diagnostic texts.

In [12]:
from tqdm import tqdm
filtered_nacc = all_nacc[list(diag_dict.keys()) + ["NACCID"]].replace({-4: np.NaN, "-4": np.NaN})
diag_texts = []
for i, row, in tqdm(filtered_nacc.iterrows()):
    st = ""
    for k, v in dict(row).items():
        if k == "NACCID":
            continue
        val = diag_dict[k][0] + diag_dict[k][1]
        try:
            if not pd.isna(v):
                st += f"{val.replace(';', ',')}: {code_dict[k][str(int(v))].replace(';', ',')}; "
        except:
            st += f"{val.replace(';', ',')}: {v}; "
        
    diag_texts.append(st)

47165it [00:19, 2376.34it/s]


In [13]:
len(diag_texts)

47165

## Data Export

- Save the diagnosis dictionary to a JSON file for further use.

In [14]:
all_nacc["diag_SUMMARY"] = diag_texts
all_nacc["patient_SUMMARY"] = patient_texts

In [15]:
all_nacc.to_csv("../../data/nacc_unique_with_summary.csv", index=False)